<a href="https://colab.research.google.com/github/JuanSc120/inteligencia-artificial-ll/blob/main/TareaAnalisisDeDatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Para este análisis avanzado, y siguiendo la fase de modelado de la metodología CRISP-DM, utilizaremos el Dataset de Viviendas de California (el estándar moderno que reemplazó a Boston Housing). Compararemos un modelo basado en algoritmos Kernel (Support Vector Regressor) contra un modelo de ensamble (Random Forest Regressor), integrando además un paso esencial de Knowledge Discovery in Databases (KDD): el escalado de características.

# Análisis Predictivo Avanzado: Comparación de Ensembles (Random Forest) vs Algoritmos Basados en Kernels (SVR)
Este notebook tiene como objetivo construir y comparar dos modelos de regresión avanzados para predecir el valor medio de las viviendas en California (MedHouseVal). Implementaremos un modelo de ensamble (Random Forest) y una Máquina de Vectores de Soporte para Regresión utilizando un kernel RBF (Radial Basis Function).
Además, introduciremos el escalado de características, un paso crítico para garantizar el rendimiento de los modelos paramétricos y geométricos.

In [1]:
# Importo pandas para la manipulación de dataframes y manipulación de datos estructurados
import pandas as pd

# Utilizo el dataset de California Housing directamente desde el módulo de datasets de scikit-learn
from sklearn.datasets import fetch_california_housing

# Cargo los datos directamente en memoria
california = fetch_california_housing()

# Convierto los arreglos de numpy a un DataFrame de pandas para facilitar su manipulación y exploración
df = pd.DataFrame(california.data, columns=california.feature_names)

# Añado la variable objetivo (target) al DataFrame.
# En este dataset, representa el valor medio de la vivienda en cientos de miles de dólares ($100,000)
df['MedHouseVal'] = california.target

# Muestro las primeras 5 filas para comprobar que la estructura, columnas y valores cargaron bien
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [2]:
# Reviso la cantidad de filas y columnas que contiene el conjunto de datos
print("Dimensiones del dataset:", df.shape)

# Verifico si existen valores nulos por columna para saber si requiero procesos de imputación de datos
print("\nValores nulos por columna:\n", df.isnull().sum())

Dimensiones del dataset: (20640, 9)

Valores nulos por columna:
 MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64


In [3]:
# Elijo la variable objetivo a predecir
y = df['MedHouseVal']

# Selecciono las variables predictoras (features); descarto la columna objetivo para no sesgar el modelo
X = df.drop(columns=['MedHouseVal'])

# Imprimo las características seleccionadas para confirmar la exclusión de la variable respuesta
print("Variables predictoras seleccionadas:", list(X.columns))

Variables predictoras seleccionadas: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


In [4]:
# Importo train_test_split para dividir la muestra
from sklearn.model_selection import train_test_split
# Importo StandardScaler para la estandarización de las variables
from sklearn.preprocessing import StandardScaler

# Divido el 80% de los registros para entrenar y reservo el 20% para evaluar el rendimiento general
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Instancio el escalador. Los algoritmos basados en Kernels (como SVR) calculan distancias entre puntos,
# por lo que variables en distintas escalas (ej. edad vs ingresos) pueden arruinar el modelo.
scaler = StandardScaler()

# Ajusto (fit) el escalador SOLO con los datos de entrenamiento para evitar fugas de información (data leakage),
# y luego transformo tanto el conjunto de entrenamiento como el de prueba.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Imprimo la cantidad de filas en cada bloque
print(f"Datos de entrenamiento: {X_train_scaled.shape[0]} registros | Datos de prueba: {X_test_scaled.shape[0]} registros")

Datos de entrenamiento: 16512 registros | Datos de prueba: 4128 registros


In [5]:
# Importo el estimador de Máquinas de Vectores de Soporte para regresión
from sklearn.svm import SVR

# Instancio el modelo SVR utilizando un kernel radial (RBF).
# El kernel RBF permite mapear los datos a un espacio de dimensión infinita para encontrar relaciones no lineales complejas.
# C=1.0 es el parámetro de regularización, epsilon=0.1 define el margen de tolerancia donde no se penalizan errores.
modelo_svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)

# Entreno el modelo basado en kernels utilizando los datos de entrenamiento escalados
modelo_svr.fit(X_train_scaled, y_train)

SVR()

In [6]:
# Importo el estimador de ensamble de Árboles Aleatorios
from sklearn.ensemble import RandomForestRegressor

# Instancio el modelo de Random Forest.
# n_estimators=100 crea un "bosque" de 100 árboles de decisión independientes.
# max_depth=15 limita la profundidad para evitar el sobreajuste (overfitting).
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)

# Entreno el ensamble. Al estar basado en árboles, este modelo particiona el espacio de características
# y no es estrictamente dependiente del escalado, pero usamos los datos escalados para mantener la igualdad de condiciones en la prueba.
modelo_rf.fit(X_train_scaled, y_train)

RandomForestRegressor(max_depth=15, random_state=42)

In [7]:
# Importo las métricas estándar de evaluación para problemas de regresión
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Genero las predicciones con el modelo SVR sobre los datos de prueba
pred_svr = modelo_svr.predict(X_test_scaled)

# Genero las predicciones con el modelo Random Forest sobre los mismos datos de prueba
pred_rf = modelo_rf.predict(X_test_scaled)

In [8]:
# ----- Evaluación del Modelo Support Vector Regressor (Kernel) -----
mae_svr = mean_absolute_error(y_test, pred_svr)
rmse_svr = np.sqrt(mean_squared_error(y_test, pred_svr))
r2_svr = r2_score(y_test, pred_svr)

# ----- Evaluación del Modelo Random Forest -----
mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
r2_rf = r2_score(y_test, pred_rf)

# Construyo un dataframe estructurado para tabular y contrastar el desempeño de ambos modelos algorítmicos
tabla_comparativa = pd.DataFrame({
    "Métrica": ["MAE (Error Medio Absoluto)", "RMSE (Raíz Error Cuadrático)", "R² (Varianza Explicada)"],
    "SVR (Kernel RBF)": [round(mae_svr, 4), round(rmse_svr, 4), round(r2_svr, 4)],
    "Random Forest": [round(mae_rf, 4), round(rmse_rf, 4), round(r2_rf, 4)]
})

# Imprimo la tabla consolidada
display(tabla_comparativa)

,Métrica,SVR (Kernel RBF),Random Forest
0,MAE (Error Medio Absoluto),0.3986,0.3332
1,RMSE (Raíz Error Cuadrático),0.5975,0.5113
2,R² (Varianza Explicada),0.7276,0.8005


### Análisis Comparativo de Rendimiento de Modelos Avanzados

Al analizar las métricas resultantes de la validación cruzada sobre el conjunto de pruebas (`X_test`), extraemos las siguientes conclusiones algorítmicas:

1. **Error Absoluto Medio (MAE):**
   El modelo de ensamble **Random Forest** suele presentar un MAE notablemente menor frente al modelo basado en Kernels (SVR). Esto significa que la desviación promedio (en cientos de miles de dólares) de las predicciones del bosque aleatorio respecto al precio real es más baja, mitigando el error continuo en la estimación del valor de las propiedades.

2. **Raíz del Error Cuadrático Medio (RMSE):**
   El RMSE castiga severamente los errores grandes o valores atípicos (outliers). El **Random Forest** demuestra una robustez superior al tener un RMSE más contenido. Aunque el modelo **SVR con Kernel RBF** es excelente proyectando hiperplanos en dimensiones superiores para separar datos, en este conjunto específico, la variabilidad de precios y la densidad poblacional (variables altamente asimétricas) son capturadas de manera más eficiente por los cortes binarios recursivos de los múltiples árboles del ensamble.

3. **R² (Varianza Explicada):**
   El coeficiente de determinación (R²) confirma la superioridad del ensamble en este contexto. Un R² más cercano a 1 indica que el Random Forest logra explicar una porción mucho más alta de la varianza en los precios de las casas a partir de las características geográficas y demográficas aportadas. El SVR logra un modelado predictivo decente, pero sufre más si los hiperparámetros (`C` y `gamma`) no pasan por un proceso exhaustivo de *GridSearch* o *RandomizedSearch*.

**Conclusión Metodológica:**
Para problemas estructurales multivariados donde interactúan variables continuas con distribuciones complejas (como ingresos económicos vs latitud/longitud espacial), los enfoques de **Ensamble (Bagging)** como el Random Forest superan la configuración por defecto de los modelos puramente geométricos o basados en distancia como el SVR. La capacidad del bosque para promediar el error de 100 estimadores débiles independientes reduce drásticamente la varianza del modelo final, consolidándose como la opción más viable para el despliegue de esta arquitectura de predicción.